In [2]:
import wikipedia

# Target your specific page
page_title = "Artificial Intelligence"
page = wikipedia.page(page_title)

# Save the clean text directly to a .txt file
with open(f"{page_title}.txt", "w", encoding="utf-8") as file:
    file.write(page.content)


In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np, re

model = SentenceTransformer("all-MiniLM-L6-v2")
text = open("Artificial Intelligence.txt", encoding="utf-8").read()
tok = model.tokenizer
def n_tokens(s): return len(tok.encode(s, add_special_tokens=False))
print(model.max_seq_length, n_tokens(text))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (17282 > 256). Running this sequence through the model will result in indexing errors


256 17282


In [7]:
def chunk_fixed(text, size=120, overlap=30):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size - overlap)]

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def chunk_sentences(text, max_tokens=150, overlap_sents=1):
    chunks, cur, cur_tok = [], [], 0
    for s in split_sentences(text):
        t = n_tokens(s)
        if cur and cur_tok + t > max_tokens:
            chunks.append(" ".join(cur))
            cur = cur[-overlap_sents:] if overlap_sents else []
            cur_tok = sum(n_tokens(x) for x in cur)
        cur.append(s); cur_tok += t
    if cur: chunks.append(" ".join(cur))
    return chunks

def chunk_recursive(text, max_tokens=150, seps=("\n\n", "\n", ". ", " ")):
    if n_tokens(text) <= max_tokens or not seps:
        return [text]
    sep, rest = seps[0], seps[1:]
    chunks, cur = [], ""
    for p in text.split(sep):
        cand = (cur + sep + p) if cur else p
        if n_tokens(cand) <= max_tokens:
            cur = cand
        else:
            if cur: chunks.append(cur)
            if n_tokens(p) > max_tokens:
                chunks.extend(chunk_recursive(p, max_tokens, rest))
                cur = ""
            else:
                cur = p
    if cur: chunks.append(cur)
    return chunks

In [8]:
questions = [
    ("In what year did AI become a formal academic field?", "academic discipline in 1956"),
    ("What hardware change around 2012 helped deep learning take off?", "started being used to accelerate neural networks"),
    ("After which year did progress speed up because of the transformer design?", "accelerated further after 2017"),
    ("What issue made early reasoning programs slow down drastically as problems grew?", "combinatorial explosion"),
    ("Which simple classifier is reportedly the favourite learner at Google because it scales well?", "most widely used learner"),
    ("How many hidden layers make a network count as deep?", "at least 2 hidden layers"),
    ("Which chess world champion lost to a computer in 1997?", "Garry Kasparov"),
    ("Which Go champion did AlphaGo beat in March 2016?", "Lee Sedol"),
    ("By how much did machine learning cut the cost of screening Parkinson's drug candidates?", "reduce the cost by a thousand-fold"),
    ("How much was spent on AI-made campaign content in India's 2024 elections?", "US$50 million"),
    ("Which odd visual cue did a skin-disease classifier wrongly rely on?", "include a ruler to show the scale"),
    ("What error rate was identical across races in the COMPAS system?", "exactly 61%"),
    ("Which shut-down nuclear plant did Microsoft agree to restart for its data centres?", "Three Mile Island"),
    ("How much more electricity does a ChatGPT query use than a Google search?", "10 times as much electrical energy"),
    ("Where was the first global AI safety summit held?", "Bletchley Park"),
    ("On what date did the EU's AI law come into force?", "1 August 2024"),
]


In [9]:
def hit_at_k(chunks, questions, k=3):
    embs = model.encode(chunks, normalize_embeddings=True)
    hits = 0
    for q, answer in questions:
        qv = model.encode(q, normalize_embeddings=True)
        top = np.argsort(embs @ qv)[::-1][:k]
        if any(answer.lower() in chunks[i].lower() for i in top):
            hits += 1
    return hits / len(questions)

for name, chunks in {"fixed": chunk_fixed(text),
                     "sentence": chunk_sentences(text),
                     "recursive": chunk_recursive(text)}.items():
    print(name, len(chunks), max(n_tokens(c) for c in chunks), hit_at_k(chunks, questions))

fixed 147 190 0.9375
sentence 171 255 0.8125
recursive 201 150 0.875


In [10]:
sent = chunk_sentences(text)
for c in sent:
    if n_tokens(c) > 200:
        print(n_tokens(c), c[:120])

240 Some other researchers were more optimistic. AI pioneer Jürgen Schmidhuber did not sign the joint statement, emphasising
255 AI pioneer Jürgen Schmidhuber did not sign the joint statement, emphasising that in 95% of all cases, AI research is abo


In [11]:
def diagnose(chunks, questions, k=3):
    embs = model.encode(chunks, normalize_embeddings=True)
    for q, ans in questions:
        qv = model.encode(q, normalize_embeddings=True)
        order = np.argsort(embs @ qv)[::-1]
        holders = [i for i, c in enumerate(chunks) if ans.lower() in c.lower()]
        if not holders:
            print("SPLIT |", ans)   # phrase cut in half by a chunk boundary
            continue
        best_rank = min(int(np.where(order == h)[0][0]) for h in holders) + 1
        if best_rank > k:
            print(f"MISS  | rank {best_rank} | {q}")

for name, chunks in {"fixed": chunk_fixed(text),
                     "sentence": chunk_sentences(text),
                     "recursive": chunk_recursive(text)}.items():
    print("==", name)
    diagnose(chunks, questions)

== fixed
MISS  | rank 5 | What hardware change around 2012 helped deep learning take off?
== sentence
MISS  | rank 7 | In what year did AI become a formal academic field?
MISS  | rank 4 | What hardware change around 2012 helped deep learning take off?
MISS  | rank 4 | After which year did progress speed up because of the transformer design?
== recursive
MISS  | rank 15 | In what year did AI become a formal academic field?
MISS  | rank 4 | What hardware change around 2012 helped deep learning take off?


In [12]:
chunks = chunk_sentences(text)
for i, c in enumerate(chunks):
    if "started being used to accelerate neural networks" in c:
        print(i, n_tokens(c))
        print(c)

2 142
AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields. Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human. Artificial intelligence was founded as an academic discipline in 1956. The field went through multiple cycles of optimism throughout its history, followed by periods of disappointment and loss of funding, known as AI winters. Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques. This growth accelerated further after 2017 with the transformer architecture.


In [13]:
chunks = chunk_sentences(text)
embs = model.encode(chunks, normalize_embeddings=True)
q = "What hardware change around 2012 helped deep learning take off?"
qv = model.encode(q, normalize_embeddings=True)
for i in np.argsort(embs @ qv)[::-1][:3]:
    print(i, round(float(embs[i] @ qv), 3), chunks[i][:200].replace("\n", " "))
    print()

37 0.746 Starting in 2012, the speed of deep learning was increased one hundred-fold by switching to GPUs, and there were enormous amounts of data available on the internet (called at the time "big data") as w

36 0.691 For example, in image processing, lower layers may identify edges, while higher layers may identify the concepts relevant to a human such as digits, letters, or faces. Deep learning has been used to i

137 0.602 However, several academic researchers became concerned that AI was no longer pursuing its original goal of creating versatile, fully intelligent machines. Beginning around 2002, they founded the subfi



In [14]:
questions = [(q, [p]) for q, p in questions]

# 2012 question: add the second place the article says it
questions[1][1].append("increased one hundred-fold by switching to GPUs")

In [15]:
def top3(q, chunks):
    embs = model.encode(chunks, normalize_embeddings=True)
    qv = model.encode(q, normalize_embeddings=True)
    for i in np.argsort(embs @ qv)[::-1][:3]:
        print(i, round(float(embs[i] @ qv), 3), chunks[i][:250].replace("\n", " "))
        print()

chunks = chunk_sentences(text)
top3("In what year did AI become a formal academic field?", chunks)
top3("After which year did progress speed up because of the transformer design?", chunks)

130 0.678 They developed several areas of research that would become part of AI, such as McCulloch and Pitts design for "artificial neurons" in 1943, and Turing's influential 1950 paper 'Computing Machinery and Intelligence', which introduced the Turing test a

136 0.672 AI gradually restored its reputation in the late 1990s and early 21st century by exploiting formal mathematical methods and by finding specific solutions to specific problems. This "narrow" and "formal" focus allowed researchers to produce verifiable

131 0.626 Many of the workshop attendees became the leaders of AI research in the 1960s. They and their students produced programs that the press described as "astonishing": computers were learning checkers strategies, solving word problems in algebra, proving

37 0.313 Starting in 2012, the speed of deep learning was increased one hundred-fold by switching to GPUs, and there were enormous amounts of data available on the internet (called at the time "big data") as well a

In [16]:
def hit_at_k(chunks, questions, k=3):
    embs = model.encode(chunks, normalize_embeddings=True)
    hits = 0
    for q, phrases in questions:
        qv = model.encode(q, normalize_embeddings=True)
        top = np.argsort(embs @ qv)[::-1][:k]
        if any(p.lower() in chunks[i].lower() for i in top for p in phrases):
            hits += 1
    return hits / len(questions)

In [17]:
for name, chunks in {"fixed": chunk_fixed(text),
                     "sentence": chunk_sentences(text),
                     "recursive": chunk_recursive(text)}.items():
    print(name, len(chunks), max(n_tokens(c) for c in chunks), hit_at_k(chunks, questions))

fixed 147 190 1.0
sentence 171 255 0.875
recursive 201 150 0.9375
